# S3 J6 - Context Engineering

# Objectifs

## Comprendre et maîtriser :

- Context Engineering (concept central des agents modernes)
- Construction du contexte envoyé aux LLM
- Sélection dynamique des informations
- Compression / résumé de contexte
- Windowing (fenêtre de contexte)
- Memory vs Context vs State
- Stratégies de réduction des tokens
- Qualité des réponses LLM
- Optimisation coût / performance

# Context Engineering

Objectif :

Comprendre comment construire
le contexte optimal envoyé à un LLM.

# Définition

Context Engineering =

"Sélectionner, structurer et optimiser
les informations envoyées au modèle."

# Pourquoi c'est critique ?

Un LLM ne "se souvient pas".

Il reçoit un contexte à chaque appel.
```
User
  ↓
Agent
  ↓
Conversation Messages
  ↓
Context Builder
  ↓
LLM
  ↓
Réponse
```

# Problème réel

Tu ne peux pas envoyer :

- toute la conversation
- toute la base de données
- toute la mémoire

# Limite des tokens

Chaque modèle a une fenêtre :

ex :

- 8k tokens
- 32k tokens
- 128k tokens

# Objectif du Context Engineering

Maximiser :

- pertinence
- précision
- cohérence

Minimiser :

- tokens
- bruit

# Architecture globale
```
Memory (PostgreSQL)
        ↓
State (Redis)
        ↓
Context Builder
        ↓
LLM
```
# Exemple concret

Utilisateur :

"Prépare un voyage à Rome"

# Mauvais contexte

- toute l'historique
- toutes les préférences
- tous les messages

# Bon contexte

- destination = Rome
- budget = 1200€
- dates = juin
- préférences = hôtel centre-ville

Chaque agent reçoit un contexte différent.
```
Planner
    ↓
Seulement le plan.
```
```
Researcher
    ↓
Documents
    ↓
Recherche
```
```
Writer
    ↓
Résumé
    ↓
Résultats
```

# Construction du contexte
### Les 7 sources de contexte

C'est ici que je modifierais complètement le notebook.
```
             Context Builder

                    │

 ┌──────────────────┼────────────────────┐

 ▼                  ▼                    ▼

Messages        Shared State       Long Memory

 ▼                  ▼                    ▼

Tool Results     RAG Docs       Runtime Metadata

                    ▼

             System Prompt
```
Étapes :

1. Identifier l'intention
2. Charger le state
3. Charger la mémoire utile
4. Filtrer
5. Structurer

### Source 1

Conversation Messages

Exemple :
```
messages = [
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
]
```
Expliquer :

Pourquoi il faut les conserver.

Pourquoi il ne faut pas tout conserver.

Sélection des messages

Exemple :

```
def select_recent_messages(
    messages,
    n=10
):
    return messages[-n:]
```
Résumé de conversation

Au lieu de :
```
100 messages
↓
Résumé
↓
5 messages
```
### Source 2
State

Redis
```
state = {

    "destination": "Rome",

    "budget": 1200,

    "step": "hotel"
}
```
### Source 3
Memory

PostgreSQL

Préférences utilisateur.
### Source 4
Tool Results

Très souvent oubliés.

Exemple :
```
tool_results = {

    "weather":

    "26°C",

    "flight_search":

    [...]
}
```
### Source 5
Documents RAG

On ajoute seulement les documents utiles.
### Source 6
Runtime Metadata

Exemple :
```
metadata = {

    "date": "...",

    "timezone": "...",

    "language": "fr",

    "remaining_budget": 0.35
}
```
### Source 7
System Prompt

Toujours présent.

Toujours envoyé.

Version complète

In [ ]:
class ContextBuilder:

    def build(

        self,

        messages,

        state,

        memory,

        tool_results,

        documents,

        metadata,

        system_prompt

    ):
        ...

Pipeline
```
Conversation
↓
Select Messages
↓
Retrieve Memory
↓
Retrieve RAG
↓
Retrieve Tool Results
↓
Merge
↓
Compress
↓
LLM
```

# Filtrage (Core Concept)

In [ ]:
def is_relevant(item, query):

    return query.lower() in item.lower()

# RAG vs Context Engineering

RAG :

→ récupère des documents

Context Engineering :

→ construit le prompt final

# Compression du contexte

In [ ]:
def compress(texts):

    return " | ".join(texts[:3])

# Résumé automatique

In [ ]:
def summarize(messages):

    return "Résumé de la conversation"

# Context Window Strategy

Stratégies :

- FIFO (First In First Out)
- Importance scoring
- Summarization
- Hybrid

In [ ]:
def select_context(messages, limit=5):

    return messages[-limit:]

# Importance scoring

In [ ]:
def score(item):

    keywords = ["budget", "objectif", "projet"]

    return sum(
        1 for k in keywords
        if k in item.lower()
    )

# Pipeline complet

In [ ]:
def context_pipeline(state, memory, messages):

    context = build_context(state, memory)

    context["messages"] = select_context(messages)

    return context

# Exemple complet

In [ ]:
state = {
    "task": "travel",
    "budget": 1200
}

memory = {
    "preferences": ["calme", "centre"]
}

messages = [
    "bonjour",
    "je veux voyager",
    "Rome"
]

context_pipeline(state, memory, messages)

# Observabilité du contexte

In [ ]:
def context_size(context):

    return len(str(context))

# Coût LLM

Plus le contexte est grand :
```
→ plus cher
→ plus lent
→ parfois moins précis
```

# Anti-patterns

❌ Envoyer toute la mémoire

❌ Envoyer toute la conversation

❌ Pas de filtrage

❌ Pas de compression

# Architecture cible
```
                 User

                  │

         Conversation Messages

                  │

                  ▼

          Conversation Store

                  │

       ┌──────────┼───────────┐

       ▼          ▼           ▼

Redis State   PostgreSQL   Tool Cache

                    │

             RAG Retrieval

                    │

                    ▼

           Context Builder

                    │

                    ▼

         OpenAI Responses API

                    │

                    ▼

               Tool Calling

                    │

                    ▼

               MCP Servers
```

# Mini Projet

Construire un Context Builder pour :

- agent de voyage
- agent support
- agent assistant personnel
  
# Exercice 1

Créer un filtre de mémoire pertinente

# Exercice 2

Implémenter un scoring de contexte

# Exercice 3

Ajouter compression du contexte

# Exercice 4

Mesurer le coût token estimé

# Exercice 5

Comparer :

- contexte brut
- contexte optimisé

# Notes d'architecte

Le Context Engineering est :

le véritable cœur des agents modernes.

Sans lui :

- coûts explosent
- qualité chute
- comportements instables

Avec lui :

- contrôle
- performance
- scalabilité

# Conclusion pédagogique

Tu es maintenant sur le noyau dur des systèmes IA modernes :
```
Agents
+ MCP
+ State (Redis)
+ Memory (PostgreSQL)
+ Context Engineering
= AI Systems réels
```

Vous touchez du doigt l'un des plus grands défis de l'ingénierie des agents IA : la **gestion de la mémoire à long terme et la pertinence du contexte**.

Renvoyer uniquement les $N$ derniers messages (méthode de la "fenêtre glissante" ou *Sliding Window*) montre vite ses limites. L'agent perd le fil conducteur si la discussion dure, oublie l'objectif initial, et vous payez des jetons (tokens) pour du bruit (des "merci", des salutations ou des digressions).

Filtrer par **nombre et par importance/intérêt** est effectivement la bonne approche conceptuelle. C'est ce qu'on appelle la **stratégie de condensation et de récupération contextuelle**.

Voici comment cela se présente sur les plans conceptuel et technique.

---

## 1. Vision Conceptuelle : Le "Filtre à Trois Niveaux"

Pour qu'un *Context Builder* soit intelligent, il doit diviser la mémoire de l'agent en trois couches distinctes, un peu comme le cerveau humain :

```
[ Flux de discussion brut ]
           │
           ▼
┌────────────────────────────────────────┐
│ 1. Les N derniers messages (Instantané)│ -> Conserve la fluidité immédiate
└────────────────────────────────────────┘
           │
           ▼
┌────────────────────────────────────────┐
│ 2. Résumé dynamique (Mémoire Working)  │ -> Condense le passé pour garder le fil
└────────────────────────────────────────┘
           │
           ▼
┌────────────────────────────────────────┐
│ 3. Recherche Vectorielle (Pertinence)  │ -> Rappelle les faits clés oubliés
└────────────────────────────────────────┘

```

### Les critères de sélection du Context Builder :

* **La Récence (Le "Quoi" immédiat) :** Les 3 ou 4 derniers messages sont indispensables pour que l'IA comprenne la grammaire de la phrase actuelle (les "il", "ce projet", etc.).
* **La Pertinence Sémantique (L'Intérêt) :** Si l'utilisateur reparle d'un sujet évoqué il y a 50 messages, le système doit être capable de "pêcher" ce message précis et de le réinjecter.
* **L'Importance Factuelle (Les Entités clés) :** Extraire les variables cruciales (ex: "Le budget est de 50k€", "Le techno-lead s'appelle Thomas") pour les fixer dans le contexte, peu importe le nombre de messages écoulés.

---

## 2. Implémentation Technique : L'Architecture du Context Builder

Pour traduire cela techniquement (en restant proche de patterns d'architecture propres aux LLM), on utilise généralement une combinaison de **Bases de données vectorielles**, de **LLM de scoring** et de **mécanismes de fenêtrage**.

### Composants clés du pipeline technique :

#### A. Le module de Recueil et Scoring (Asynchrone)

Chaque fois qu'un message est envoyé, il suit deux chemins :

1. **L'immédiat :** Il va dans la base de données transactionnelle (historique brut).
2. **L'asynchrone :** Un worker (ou un agent secondaire plus léger) analyse le message pour en évaluer "l'importance".
* *Option A (Algorithmique) :* On calcule l'embedding du message et on le stocke dans une **Vector DB** (ex: Chroma, PGVector, Pinecone).
* *Option B (LLM-driven) :* Un prompt très rapide demande à un modèle : *"Ce message contient-il une instruction ou une information cruciale à long terme ? Réponds par Oui/Non"*. Si Oui, on le marque d'un flag `is_important=true`.



#### B. Le mécanisme de Résumé (Summarization)

Dès que la taille de l'historique dépasse un certain seuil de tokens, un thread en arrière-plan prend les messages les plus anciens, demande au LLM d'en faire un résumé condensé sous forme de puces (bullet points), écrase ces messages dans le contexte actif et met à jour une variable globale `system_summary`.

#### C. L'assemblage dynamique du Contexte (Le cœur du Builder)

Au moment où l'agent doit répondre, le **Context Builder** assemble le prompt final en allant piocher dans ses différents compartiments :

```python
# Exemple conceptuel d'un Context Builder
def build_context(user_query, session_id, max_tokens=4000):
    context = ""
    
    # 1. Toujours inclure les instructions système de base
    context += get_system_instructions() 
    
    # 2. Injecter le résumé des discussions passées
    context += get_historical_summary(session_id)
    
    # 3. Recherche de "Pertinence/Intérêt" (RAG sur l'historique)
    # On cherche dans la Vector DB les messages passés liés à la requête actuelle
    relevant_past_messages = vector_search(query=user_query, session_id=session_id, limit=3)
    context += format_relevant_messages(relevant_past_messages)
    
    # 4. Injecter les N derniers messages pour la fluidité (Récence)
    recent_messages = get_last_messages(session_id, limit=4)
    context += format_recent_messages(recent_messages)
    
    # 5. On tronque ou ajuste si on approche de la limite de tokens
    return trim_to_max_tokens(context, max_tokens)

```

---

## 3. Les Patterns Avancés (Frameworks Agents)

Si vous développez cela vous-même ou utilisez des outils comme LangChain, LlamaIndex ou des architectures multi-agents :

* **Semantic Memory Window :** Au lieu de couper au bout de 10 messages, vous calculez la similarité cosinus entre la question actuelle et l'historique. Vous ne gardez que les messages qui ont un score de similarité > 0.7.
* **L'Agent "Concierge de Mémoire" :** Dans un système multi-agents, un agent dédié a pour unique rôle de lire le chat en continu, de mettre à jour une fiche de profils/faits sur l'utilisateur, et de la fournir à l'agent principal.

Cette approche hybride (Récence + Résumé + Recherche Vectorielle) est aujourd'hui le standard de l'industrie pour concevoir des agents capables de tenir des conversations complexes sur le long terme sans exploser le budget de tokens ni perdre en pertinence.

Souhaitez-vous que nous zoomions sur la partie vectorisation de l'historique ou sur la stratégie de résumé automatique ?


Voici comment conceptualiser et coder le Context Builder complet, capable de fusionner ces différentes dimensions de manière robuste (gestion des priorités et des limites de tokens).

---

## 1. La Vision Conceptuelle : Le "Merge Engine"

Le rôle du Context Builder est de prendre ces 7 composants (que vous avez listés) et de les fusionner dans un template final unique que le LLM pourra digérer.

Chaque composant a une **priorité** et une **politique de cycle de vie** différente :

| Composant | Nature | Priorité de conservation | Gestion des Tokens |
| --- | --- | --- | --- |
| **System Prompt** | Statique / Directives | **Critique** (100%) | Fixe, ne bouge jamais. |
| **Runtime Metadata** | Dynamique (Date, User...) | **Haute** | Très léger, souvent conservé intégralement. |
| **Tool Results** | Éphémère (Résultat d'API) | **Haute** (Juste après l'appel) | Peut être énorme (ex: un JSON de 50 lignes). Doit souvent être nettoyé ou résumé. |
| **Messages** | Évolutif (Chat History) | **Moyenne** | Soumis au fenêtrage ou au résumé. |
| **Shared State** | Mémoire de session (Variables) | **Haute** | Clés/valeurs cruciales pour la logique métier de l'agent. |
| **Long Memory** | Profil utilisateur / Faits persistants | **Moyenne** | Chargé dynamiquement selon le besoin. |
| **RAG Docs** | Connaissances externes injectées | **Variable** | Ajusté selon l'espace restant dans la fenêtre. |

---

## 2. Implémentation Technique : Pattern d'un Context Builder Robuste

Pour implémenter cela proprement (par exemple en Java/Kotlin ou en Python), on utilise un pattern de type **Builder** ou **Pipeline d'enrichissement**. Chaque composant vient enrichir un objet `ContextState`.

Voici une implémentation conceptuelle plus proche de la réalité d'un agent de production :

```python
class ContextBuilder:
    def __init__(self, token_counter, max_total_tokens=8192):
        self.token_counter = token_counter
        self.max_total_tokens = max_total_tokens

    def build(self, agent_request, runtime_env) -> list:
        """
        Prend les données brutes et assemble le payload final pour le LLM.
        Retourne généralement une liste de messages formatés (system, user, assistant, tool).
        """
        # 1. Initialisation du conteneur de messages final
        final_messages = []

        # 2. SYSTEM PROMPT & RUNTIME METADATA
        # On fusionne les règles de l'agent et le contexte d'exécution (ex: date, localisation, user_role)
        system_content = self._assemble_system_prompt(runtime_env)
        final_messages.append({"role": "system", "content": system_content})

        # 3. SHARED STATE & LONG MEMORY
        # On récupère les variables globales de la session et la mémoire long terme (ex: préférences utilisateur)
        shared_state = agent_request.shared_state
        long_memory = self._fetch_long_term_memory(agent_request.user_id)
        
        # Souvent injectés sous forme de contexte système secondaire ou de méta-données de session
        session_context = f"--- SESSION STATE ---\n{shared_state}\n--- USER PROFILE ---\n{long_memory}"
        final_messages.append({"role": "system", "content": session_context})

        # 4. RAG DOCS (Documents pertinents)
        # On va chercher les documents pertinents par rapport à la DERNIÈRE question de l'utilisateur
        if agent_request.last_user_message:
            rag_docs = self._retrieve_rag_docs(agent_request.last_user_message, limit=3)
            if rag_docs:
                final_messages.append({"role": "system", "content": f"--- RELEVANT DOCUMENTS ---\n{rag_docs}"})

        # 5. HISTORIQUE & TOOL RESULTS (Le flux chronologique)
        # C'est ici que se joue la fusion temporelle : les messages passés ET les résultats d'outils récents
        chronological_flow = self._assemble_timeline(
            history=agent_request.chat_history, 
            pending_tool_results=agent_request.tool_results
        )
        
        # 6. GESTION DU BUDGET DE TOKENS (Token Budgeting Loop)
        # On ajoute les messages du flux chronologique en partant du plus RÉCENT (le bas du schéma)
        # et on remonte jusqu'à ce que le budget de tokens soit atteint.
        budget_messages = []
        available_tokens = self.max_total_tokens - self.token_counter.count(final_messages)
        
        for msg in reversed(chronological_flow):
            msg_tokens = self.token_counter.count([msg])
            if available_tokens - msg_tokens > 0:
                budget_messages.insert(0, msg) # On réinsère au début pour garder l'ordre chronologique
                available_tokens -= msg_tokens
            else:
                # Plus de place ! On peut ici déclencher une stratégie de fallback (ex: résumer le reste)
                break
                
        # Assemblage final
        final_messages.extend(budget_messages)
        
        return final_messages

    def _assemble_system_prompt(self, runtime_env):
        base_prompt = "Tu es un agent expert..."
        metadata_str = f"Date actuelle: {runtime_env.current_date}. Pays: {runtime_env.country}."
        return f"{base_prompt}\n\n[Runtime Context]\n{metadata_str}"

    def _assemble_timeline(self, history, pending_tool_results):
        """Fusionne l'historique existant et les derniers résultats d'outils exécutés"""
        flow = list(history)
        # Si des outils viennent d'être exécutés, on les ajoute à la fin de la timeline
        for tool_res in pending_tool_results:
            flow.append({
                "role": "tool", 
                "tool_call_id": tool_res.call_id, 
                "content": tool_res.output
            })
        return flow

```

---

## 3. Les points critiques à gérer dans ce pattern

1. **Le "Sandwich" du Tool Result :** Lorsqu'un agent appelle un outil, la séquence exacte attendue par les modèles (comme GPT ou Claude) est : `User Message` -> `Assistant (Tool Call)` -> `Tool (Result)` -> `Assistant (Final Response)`. Le Context Builder doit s'assurer de ne jamais casser cette linéarité sémantique, sinon le LLM lève une erreur de protocole.
2. **L'isolation du Shared State :** Le `Shared State` (les variables d'état que les agents se passent entre eux) ne doit pas polluer l'historique de discussion de l'utilisateur. Il doit être injecté proprement, souvent sous forme de bloc de données structurées (JSON ou XML) dans la section système, pour que l'agent sache "où il en est" dans son workflow global.
3. **Le dilemme du RAG vs Long Memory :** * La *Long Memory* répond à : *Qui est l'utilisateur ?* (Données stables).
* Le *RAG* répond à : *De quoi parle-t-on ?* (Données externes).
Le Builder doit savoir calibrer la place accordée à chacun. Si le RAG renvoie un document massif, il risque d'étouffer les messages récents ou le Shared State. Une troncature intelligente ou un re-ranking (via un modèle Cross-Encoder) est indispensable avant l'injection dans le Builder.